### How I Prepared Data for the Lesson

In this extra content notebook, I'm sharing how I prepped the data we will be using in class. This could be a good reference for your project #2 but you miught find that the tools you will learn will simplify data prep even more! If you have any questions about what I've done below, please email me!

##### The Data I'm Working With

Pokemon Stats Dataset : [Kaggle Pokemon Dataset](https://www.kaggle.com/datasets/rounakbanik/pokemon) <br> *Note - I removed uneeded columns from the data using good old fashioned Excel*

Pokemon Images Dataset: [Kaggle Pokemon Images DataSet](https://www.kaggle.com/datasets/hlrhegemony/pokemon-image-dataset)

##### My Output Goals

I want a JSON file of all the Pokemons' stats, nicely formatted.

I want a single folder containing images of the pokemon, where each pokemon has 1 image, and each image is named the pokemon's name.

---


### Step 1 - Importing the Libraries We Will Need
We've used the library `math` a tiny bit so far. From this point onwards, we will be using libraries a lot more!

You'll find libraries for almost any functionality you want to implement - some of them you need to download, and some are packaged with Python. 

We will only be using pre-packaged libraries in this notebook. Pre-packaged libraries belong to [Python's Standard Library](https://docs.python.org/3/library/index.html). If you click on the link, you'll see that there is A LOT included with Python. This is why Python is referred to as a *batteries included* language.

In [34]:
import csv # used to read and write CSV files
import json # used to read and write JSON files
import re # used for regex (advanced string editing)
import shutil # used for copying and renaming files
import os # used for accessing files and folders on your computer

### Step 2 - Reading the CSV File Containing the Pokemon Stats
I just wanted to note the keyword `with`.

In Python, `with` allows us to properly open a file and not have to worry about closing it once we're done. You can see [this link](https://www.geeksforgeeks.org/with-statement-in-python/) for more context. 

When we type `with open('data/pokemon.csv', mode='r', newline='', encoding='utf-8') as csvfile:` we are creating a ***context manager*** `csvfile`. A context manager allows us to automate the resource handling of a given resource - in this case the pokemon stats .csv file. We pass in arguments that define how the context manager interacts with the file. In this case `mode=r`(read), `newline=''`(our new line seperator is just a new line), and `encoding=utf-8`(standard csv encoding).

We can then create a `csv.DictReader` object by passing in the `csvfile` context manager as an argument. This `csv.DictReader` object is iterable so we can easily cast it to a list which we point the variable `pokemon` at.

The result of all this is that we open the .csv file, read it, convert the read content to a list, and then close the file automatically.

In [35]:
# Open the CSV file, convert it to a list
with open('data/pokemon.csv', mode='r', newline='', encoding='utf-8') as csvfile:
    pokemon_list = list(csv.DictReader(csvfile))

# Preview the first entry in the list
pokemon_list[0]

{'pokedex_number': '1',
 'name': 'Bulbasaur',
 'type1': 'grass',
 'type2': 'poison',
 'attack': '49',
 'defense': '49',
 'hp': '45',
 'abilities': "['Overgrow', 'Chlorophyll']",
 'against_bug': '1',
 'against_dark': '1',
 'against_dragon': '1',
 'against_electric': '0.5',
 'against_fairy': '0.5',
 'against_fight': '0.5',
 'against_fire': '2',
 'against_flying': '2',
 'against_ghost': '1',
 'against_grass': '0.25',
 'against_ground': '1',
 'against_ice': '2',
 'against_normal': '1',
 'against_poison': '1',
 'against_psychic': '2',
 'against_rock': '1',
 'against_steel': '1',
 'against_water': '0.5'}

### Step 2 - Format the Data
Our data came in as a list of dictionaries where each dictionary represents the stats of a single pokemon.

We need to clean up the data a bit to make it useable for our purposes. All the numeric data came in as strings! We need to:
* Convert all the `against_x` values to floats.
* Convert all other numeric data to integers.
* Convert the list-looking string under the key `abilities` to an actual Python list
* Lower case all text to make it easier to work with.
* Some pokemon have no second type. If that's the case, we want to replace the blank string '' with Python's `None`.

In [36]:
for mon in pokemon_list: # iterate through the list of pokemon dictionaries
    
    # let's convert the 'against_x' values first
    for key in mon.keys(): # iterate the keys of each pokemon
        if "against" in key: # if 'against' is in the key string
            mon[key] = float(mon[key]) # convert the contained string value to a float

    # convert all the string that should be integers    
    mon["pokedex_number"] = int(mon["pokedex_number"])
    mon["attack"] = int(mon["attack"])
    mon["defense"] = int(mon["defense"])
    mon["hp"] = int(mon["hp"])

    # convert the list-like string to an actual list of strings
    # note that the re.sub() function is a regex function.
    # regex is super useful for string formatting.
    # but, I always use chatGPT to write my regex expressions!
    temp_str = re.sub(r"[\[\]']", "", mon["abilities"]) # remove all brackets, and apostrophes
    temp_str = re.sub(r",\s*", ",", temp_str) # replace commas and spaces with just commas
    temp_str = temp_str.lower() # lower case all the abilities
    temp_list = temp_str.split(",") # split the abilities string into an actual list with the commas
    mon["abilities"] = temp_list # assign the new list to the 'abilities' key

    # let's lowercase the pokemon's name
    mon["name"] = mon["name"].lower()

    # finally if the second pokemon type is blank, set it to None
    if mon["type2"] == "":
        mon["type2"] = None


# preview the first entry in the pokemon list
pokemon_list[6]

{'pokedex_number': 7,
 'name': 'squirtle',
 'type1': 'water',
 'type2': None,
 'attack': 48,
 'defense': 65,
 'hp': 44,
 'abilities': ['torrent', 'rain dish'],
 'against_bug': 1.0,
 'against_dark': 1.0,
 'against_dragon': 1.0,
 'against_electric': 2.0,
 'against_fairy': 1.0,
 'against_fight': 1.0,
 'against_fire': 0.5,
 'against_flying': 1.0,
 'against_ghost': 1.0,
 'against_grass': 2.0,
 'against_ground': 1.0,
 'against_ice': 0.5,
 'against_normal': 1.0,
 'against_poison': 1.0,
 'against_psychic': 1.0,
 'against_rock': 1.0,
 'against_steel': 0.5,
 'against_water': 0.5}

Now our data is formatted neatly. What we want to do now is convert the list of pokemon to a dictionary of pokemon. 

We are doing this so we can use each Pokemon's name as a key in the dictionary. The value for each pokemon will be another dictionary containing all of it's stats.

In [37]:
# create an empty dictionary
pokemon_dict = {}

# iterate the pokemon list
for mon in pokemon_list:
    poke_name = mon.pop("name") # remove the 'name' key/value pair from the inital pokemon dictionary. Assign the removed value to the variable 'poke_name'.
    pokemon_dict[poke_name] = mon # create a key in the new dictionary thats the pokemons name. The value is the initial dictionary

# Now we can look up a value by name
pokemon_dict["squirtle"]

{'pokedex_number': 7,
 'type1': 'water',
 'type2': None,
 'attack': 48,
 'defense': 65,
 'hp': 44,
 'abilities': ['torrent', 'rain dish'],
 'against_bug': 1.0,
 'against_dark': 1.0,
 'against_dragon': 1.0,
 'against_electric': 2.0,
 'against_fairy': 1.0,
 'against_fight': 1.0,
 'against_fire': 0.5,
 'against_flying': 1.0,
 'against_ghost': 1.0,
 'against_grass': 2.0,
 'against_ground': 1.0,
 'against_ice': 0.5,
 'against_normal': 1.0,
 'against_poison': 1.0,
 'against_psychic': 1.0,
 'against_rock': 1.0,
 'against_steel': 0.5,
 'against_water': 0.5}

### Step 3 - Export the .JSON File
We again use the keyword `with` to handle the file managment. In this case however, we are creating a file instead of reading it, hence `mode='w'`. 

We use the `json.dump` function to write a .json file from our pokemon dictionary. We pass `indent=4` to make our .json file more legible.

This file will be the one we work with in class.

In [38]:
with open('data/pokemon.json', mode='w', encoding='utf-8') as jsonfile:
    json.dump(pokemon_dict, jsonfile, indent=4)

### Step 4 - Organizing Our Images
The images I downloaded were formatted as such:
* A folder `data/dl_images/` filled with subfolders where each subfolder's name was a pokemon's name
* Each subfolder contained 2-4 .jpg images of the pokemon where each image was named `0.jpg`, `1.jpg`, etc.

I want to reorganize the images like so:
* A single folder `data/images/` containing .jpg images - 1 for each pokemon.
* Each .jpg image is named as the depicted pokemon's name, all lowercase.

To do this we will reference our pokemon dictionary, and then move and rename the '0.jpg' image from the corresponding `data/dl_images` subfolder to the new folder `data/images/`.

In [39]:
old_folder = "data/dl_images/" # the folder I downloaded from kaggle
new_folder = "data/images/" # the new folder I want to copy to

for img_folder in os.listdir(old_folder): # iterate through the sub-folders in dl_images - ie. the folders with the pokemons name
    if img_folder.lower() in pokemon_dict.keys(): # check if we have the pokemon in our pokemon dictionary
        old_img_path = f"{old_folder}{img_folder}/0.jpg" # get the file path of the 0.jpg image we want to copy
        old_name_new_folder_path = shutil.copy(old_img_path, new_folder) # copy the 0.jpg image to the new folder
        new_name_new_folder_path = f"{new_folder}{img_folder.lower()}.jpg" # create the new name - aka the pokemons name all lowercase
        shutil.move(old_name_new_folder_path, new_name_new_folder_path) # rename the file
        